# 03 — Validation Notebook
**Objective:** Verify that all 20 cleaning rules were applied correctly.  
Each test loads the cleaned data and checks that the expected transformations happened.

A test **passes** if the cleaned data meets the expected condition.  
A test **fails** if any residual mess remains.

---

In [476]:
import pandas as pd
import numpy as np
import re
import hashlib

# ── Load cleaned outputs ──
DATA_DIR = '../cleaned/full'
df      = pd.read_csv(f'{DATA_DIR}/raw_cems_data_cleaned.csv')
manual  = pd.read_csv(f'{DATA_DIR}/manual_entries_cleaned.csv')
sensors = pd.read_csv(f'{DATA_DIR}/sensor_master.csv')
maint   = pd.read_csv(f'{DATA_DIR}/maintenance_logs.csv')
thresh  = pd.read_csv(f'{DATA_DIR}/regulatory_thresholds.csv')
audit   = pd.read_csv(f'{DATA_DIR}/cleaning_log.csv')

# parse TS for time-based checks
df['TS'] = pd.to_datetime(df['TS'], errors='coerce')

# ── Results tracker ──
results = []

def check(rule, description, passed, detail=''):
    """Record a validation check result."""
    status = 'PASS' if passed else 'FAIL'
    icon = '\u2705' if passed else '\u274c'
    results.append({'Rule': rule, 'Check': description, 'Status': status, 'Detail': detail})
    msg = f'{icon} {rule:8s} | {description}'
    if detail:
        msg += f' — {detail}'
    print(msg)

print(f'Loaded cleaned data: {df.shape}')
print(f'Audit log entries: {len(audit)}')
print(f'\n{"="*60}')
print('  VALIDATION CHECKS')
print(f'{"="*60}\n')

Loaded cleaned data: (28800, 14)
Audit log entries: 124306

  VALIDATION CHECKS



---
## Schema & Structure Checks

In [477]:
# ═══ Schema: Expected columns exist ═══
expected_cols = ['Record_ID', 'Plant_ID', 'Stack_ID', 'Flow_Rate_m3_hr',
                 'TS', 'PM2.5', 'SO2', 'NOx', 'Unit', 'Status', 'Lat_Lon',
                 'Source_Type', 'Exceedance_Flag', 'Audit_Hash']

missing = [c for c in expected_cols if c not in df.columns]
check('SCHEMA', 'All expected columns present',
      len(missing) == 0,
      f'Missing: {missing}' if missing else f'{len(expected_cols)} columns verified')

# ═══ Data types ═══
pollutant_numeric = all(pd.api.types.is_float_dtype(df[c]) for c in ['PM2.5','SO2','NOx'])
check('SCHEMA', 'Pollutant columns are float64',
      pollutant_numeric,
      f'PM2.5={df["PM2.5"].dtype}, SO2={df["SO2"].dtype}, NOx={df["NOx"].dtype}')

ts_datetime = pd.api.types.is_datetime64_any_dtype(df['TS'])
check('SCHEMA', 'TS is datetime64',
      ts_datetime,
      f'dtype={df["TS"].dtype}')

✅ SCHEMA   | All expected columns present — 14 columns verified
✅ SCHEMA   | Pollutant columns are float64 — PM2.5=float64, SO2=float64, NOx=float64
✅ SCHEMA   | TS is datetime64 — dtype=datetime64[ns]


---
## Rule-by-Rule Validation

In [478]:
# ═══ R1: No midnight rollover timestamps (24:xx) ═══
# We check the raw string before parsing — since TS is already datetime,
# if it parsed successfully, 24:xx was fixed.
ts_nulls = df['TS'].isna().sum()
gap_filled = (df['Status'] == 'GAP_FILLED').sum()
# TS nulls should only be from gap-filled rows
check('R1', 'No midnight rollover (24:xx) — all timestamps parsed',
      ts_nulls == gap_filled,
      f'TS nulls={ts_nulls}, gap_filled={gap_filled} (should match)')

# Also verify audit log has R1 entries
r1_logs = len(audit[audit['Rule'] == 'R1'])
check('R1', 'Audit log has midnight rollover entries',
      r1_logs > 0,
      f'{r1_logs} changes logged')

✅ R1       | No midnight rollover (24:xx) — all timestamps parsed — TS nulls=2415, gap_filled=2415 (should match)
✅ R1       | Audit log has midnight rollover entries — 797 changes logged


In [479]:
# ═══ R2: No unicode unit strings remaining ═══
unicode_units = df['Unit'].isin(['\u00b5g/m\u00b3']).sum()
check('R2', 'No unicode \u00b5g/m\u00b3 remaining',
      unicode_units == 0,
      f'Found: {unicode_units}')

r2_logs = len(audit[audit['Rule'] == 'R2'])
check('R2', 'Audit log has unit standardization entries',
      r2_logs > 0,
      f'{r2_logs} changes logged')

✅ R2       | No unicode µg/m³ remaining — Found: 0
✅ R2       | Audit log has unit standardization entries — 2623 changes logged


In [480]:
# ═══ R3: Status values are trimmed and uppercased ═══
has_whitespace = df['Status'].str.contains(r'^\s|\s$', regex=True, na=False).sum()
has_lowercase = df['Status'].apply(lambda s: s != s.upper() if isinstance(s, str) else False).sum()
check('R3', 'No leading/trailing whitespace in Status',
      has_whitespace == 0,
      f'Found: {has_whitespace}')
check('R3', 'All Status values are uppercase',
      has_lowercase == 0,
      f'Found: {has_lowercase}')

✅ R3       | No leading/trailing whitespace in Status — Found: 0
✅ R3       | All Status values are uppercase — Found: 0


In [481]:
# ═══ R4: Coordinates valid (India bounds) ═══
LAT_MIN, LAT_MAX = 6.0, 37.0
LON_MIN, LON_MAX = 68.0, 97.5

invalid_coords = 0
semicolons = 0
for val in df['Lat_Lon'].dropna():
    s = str(val)
    if ';' in s:
        semicolons += 1
    try:
        parts = s.split(',')
        lat, lon = float(parts[0]), float(parts[1])
        if lat < LAT_MIN or lat > LAT_MAX or lon < LON_MIN or lon > LON_MAX:
            invalid_coords += 1
    except:
        invalid_coords += 1

check('R4', 'No semicolons in Lat_Lon',
      semicolons == 0,
      f'Found: {semicolons}')
check('R4', 'All coordinates within India bounds',
      invalid_coords == 0,
      f'Invalid: {invalid_coords}')

✅ R4       | No semicolons in Lat_Lon — Found: 0
✅ R4       | All coordinates within India bounds — Invalid: 0


In [482]:
# ═══ R5: No negative pollutant values ═══
for col in ['PM2.5', 'SO2', 'NOx']:
    neg_count = (df[col] < 0).sum()
    check('R5', f'No negatives in {col}',
          neg_count == 0,
          f'Found: {neg_count}')

✅ R5       | No negatives in PM2.5 — Found: 0
✅ R5       | No negatives in SO2 — Found: 0
✅ R5       | No negatives in NOx — Found: 0


In [483]:
# ═══ R6: No duplicates (Plant_ID + Stack_ID + TS) ═══
non_gap = df[df['Status'] != 'GAP_FILLED']  # gap-filled have NaT timestamps
dups = non_gap.duplicated(subset=['Plant_ID', 'Stack_ID', 'TS'], keep=False).sum()
check('R6', 'No duplicate rows (Plant+Stack+TS)',
      dups == 0,
      f'Duplicates: {dups}')

✅ R6       | No duplicate rows (Plant+Stack+TS) — Duplicates: 0


In [484]:
# ═══ R7: Calibration was applied (audit log check) ═══
r7_logs = len(audit[audit['Rule'] == 'R7'])
check('R7', 'Calibration adjustments were made',
      r7_logs > 0,
      f'{r7_logs} adjustments logged')

✅ R7       | Calibration adjustments were made — 73900 adjustments logged


In [485]:
# ═══ R8: No physical spikes > 5000 remain ═══
for col in ['PM2.5', 'SO2', 'NOx']:
    over_5000 = (df[col] > 5000).sum()
    check('R8', f'No spikes > 5000 in {col}',
          over_5000 == 0,
          f'Found: {over_5000}, max={df[col].max():.1f}')

✅ R8       | No spikes > 5000 in PM2.5 — Found: 0, max=2499.9
✅ R8       | No spikes > 5000 in SO2 — Found: 0, max=2499.9
✅ R8       | No spikes > 5000 in NOx — Found: 0, max=2550.8


In [486]:
# ═══ R9: Record_IDs are continuous (E00001 to E0XXXX) ═══
record_nums = df['Record_ID'].str.replace('E', '').astype(int)
expected_range = set(range(record_nums.min(), record_nums.max() + 1))
actual_range = set(record_nums)
gaps = expected_range - actual_range

check('R9', 'Record_IDs are continuous (no gaps)',
      len(gaps) == 0,
      f'Missing IDs: {len(gaps)}')

# verify gap-filled rows exist and have correct status
gap_rows = df[df['Status'] == 'GAP_FILLED']
gap_all_nan = gap_rows[['PM2.5', 'SO2', 'NOx']].isna().all().all()
check('R9', 'GAP_FILLED rows have NaN pollutant values',
      gap_all_nan,
      f'{len(gap_rows)} gap-filled rows')

✅ R9       | Record_IDs are continuous (no gaps) — Missing IDs: 0
✅ R9       | GAP_FILLED rows have NaN pollutant values — 2415 gap-filled rows


In [487]:
# ═══ R10: Maintenance windows respected ═══
maint_parsed = maint.copy()
maint_parsed['Maint_Start'] = pd.to_datetime(maint_parsed['Maint_Start'])
maint_parsed['Maint_End'] = pd.to_datetime(maint_parsed['Maint_End'])

maint_violations = 0
for _, m in maint_parsed.iterrows():
    during_maint = df[
        (df['Plant_ID'] == m['Plant_ID']) &
        (df['Stack_ID'] == m['Stack_ID']) &
        (df['TS'] >= m['Maint_Start']) &
        (df['TS'] <= m['Maint_End'])
    ]
    # these rows should all have Status=MAINT and NaN pollutants
    not_maint = during_maint[during_maint['Status'] != 'MAINT']
    maint_violations += len(not_maint)

check('R10', 'All rows during maintenance have Status=MAINT',
      maint_violations == 0,
      f'Violations: {maint_violations}')

✅ R10      | All rows during maintenance have Status=MAINT — Violations: 0


In [488]:
# ═══ R11: No BDL strings remaining ═══
for col in ['PM2.5', 'SO2', 'NOx']:
    str_vals = df[col].astype(str).str.strip().str.upper()
    bdl_remaining = str_vals.isin(['BDL']).sum() + str_vals.str.startswith('<').sum()
    check('R11', f'No BDL strings in {col}',
          bdl_remaining == 0,
          f'Found: {bdl_remaining}')

✅ R11      | No BDL strings in PM2.5 — Found: 0
✅ R11      | No BDL strings in SO2 — Found: 0
✅ R11      | No BDL strings in NOx — Found: 0


In [489]:
# ═══ R12: Only canonical status values ═══
canonical = {'OK', 'FAULT', 'MAINT', 'GAP_FILLED', 'UNKNOWN'}
actual_statuses = set(df['Status'].unique())
non_canonical = actual_statuses - canonical

check('R12', 'Only canonical status values exist',
      len(non_canonical) == 0,
      f'Non-canonical: {non_canonical}' if non_canonical else f'Values: {actual_statuses}')

✅ R12      | Only canonical status values exist — Values: {'UNKNOWN', 'MAINT', 'OK', 'GAP_FILLED', 'FAULT'}


In [490]:
# ═══ R13: All Plant+Stack combos are valid ═══
valid_combos = set(zip(sensors['Plant_ID'], sensors['Stack_ID']))
data_combos = set(zip(df['Plant_ID'].dropna(), df['Stack_ID'].dropna()))
invalid_combos = data_combos - valid_combos

check('R13', 'All Plant+Stack combos exist in sensor_master',
      len(invalid_combos) == 0,
      f'Invalid: {invalid_combos}' if invalid_combos else 'All valid')

✅ R13      | All Plant+Stack combos exist in sensor_master — All valid


In [491]:
# ═══ R14: No mg/Nm3 remaining ═══
mg_remaining = (df['Unit'] == 'mg/Nm3').sum()
check('R14', 'No mg/Nm3 unit remaining',
      mg_remaining == 0,
      f'Found: {mg_remaining}')

# all units should be ug/m3
all_ugm3 = (df['Unit'] == 'ug/m3').all()
check('R14', 'All units are ug/m3',
      all_ugm3,
      f'Values: {df["Unit"].value_counts().to_dict()}')

✅ R14      | No mg/Nm3 unit remaining — Found: 0
✅ R14      | All units are ug/m3 — Values: {'ug/m3': 28800}


In [492]:
# ═══ R15: Source_Type column exists and is valid ═══
has_source_type = 'Source_Type' in df.columns
check('R15', 'Source_Type column exists',
      has_source_type,
      f'Values: {df["Source_Type"].value_counts().to_dict()}' if has_source_type else 'MISSING')

valid_types = {'Stack', 'Ambient', 'Unknown'}
actual_types = set(df['Source_Type'].unique()) if has_source_type else set()
check('R15', 'Source_Type has valid values only',
      actual_types.issubset(valid_types),
      f'Values: {actual_types}')

✅ R15      | Source_Type column exists — Values: {'Stack': 15801, 'Ambient': 10584, 'Unknown': 2415}
✅ R15      | Source_Type has valid values only — Values: {'Unknown', 'Ambient', 'Stack'}


In [493]:
# ═══ R16: QC_Status column in manual entries ═══
has_qc = 'QC_Status' in manual.columns
check('R16', 'QC_Status column exists in manual entries',
      has_qc,
      f'Values: {manual["QC_Status"].value_counts().to_dict()}' if has_qc else 'MISSING')

if has_qc:
    # verify QC logic: Diff_Pct > 1 should be QC_FAIL
    qc_correct = True
    for _, row in manual.iterrows():
        expected = 'QC_FAIL' if row['Diff_Pct'] > 1.0 else 'QC_PASS'
        if row['QC_Status'] != expected:
            qc_correct = False
            break
    check('R16', 'QC_Status correctly flags >1% differences',
          qc_correct, 'Logic verified')

✅ R16      | QC_Status column exists in manual entries — Values: {'QC_PASS': 40, 'QC_FAIL': 10}
✅ R16      | QC_Status correctly flags >1% differences — Logic verified


In [494]:
# ═══ R17: No UTC timestamps remaining ═══
# Since TS is datetime, we can't check for "UTC" string directly.
# But we verify via audit log that conversions happened.
r17_logs = len(audit[audit['Rule'] == 'R17'])
check('R17', 'UTC-to-IST conversions were made',
      r17_logs > 0,
      f'{r17_logs} conversions logged')

# Also check date range is reasonable (IST, not UTC)
valid_ts = df['TS'].dropna()
date_range_ok = valid_ts.min().year == 2025
check('R17', 'Date range is in 2025 (post-conversion)',
      date_range_ok,
      f'{valid_ts.min()} to {valid_ts.max()}')

✅ R17      | UTC-to-IST conversions were made — 1292 conversions logged
✅ R17      | Date range is in 2025 (post-conversion) — 2025-01-01 00:00:00 to 2025-01-30 23:45:00


In [495]:
# ═══ R18: No PII in inspection notes ═══
phone_pattern = r'\b\d{10}\b'
email_pattern = r'\S+@\S+\.\S+'

pii_found = 0
for note in manual['Inspection_Notes'].dropna():
    note_str = str(note)
    # skip if already redacted
    clean_note = note_str.replace('[REDACTED_PHONE]', '').replace('[REDACTED_EMAIL]', '')
    if re.search(phone_pattern, clean_note):
        pii_found += 1
    if re.search(email_pattern, clean_note):
        pii_found += 1

check('R18', 'No unredacted PII in Inspection_Notes',
      pii_found == 0,
      f'Unredacted PII: {pii_found}')

# verify redaction happened
redacted = manual['Inspection_Notes'].str.contains('REDACTED', na=False).sum()
check('R18', 'Redacted notes exist (proving PII was found & fixed)',
      redacted > 0,
      f'{redacted} notes contain [REDACTED]')

✅ R18      | No unredacted PII in Inspection_Notes — Unredacted PII: 0
✅ R18      | Redacted notes exist (proving PII was found & fixed) — 16 notes contain [REDACTED]


In [496]:
# ═══ R19: Exceedance_Flag column is correct ═══
has_flag = 'Exceedance_Flag' in df.columns
check('R19', 'Exceedance_Flag column exists',
      has_flag,
      f'Values: {df["Exceedance_Flag"].value_counts().to_dict()}' if has_flag else 'MISSING')

if has_flag:
    # spot-check: verify a few exceedances are correct
    limit_lookup = {}
    for _, t in thresh.iterrows():
        limit_lookup[(t['Pollutant'], t['Source_Type'])] = t['Legal_Limit_ugm3']
    
    # sample 100 rows and verify
    sample = df[df['Source_Type'] != 'Unknown'].sample(min(100, len(df)), random_state=42)
    mismatches = 0
    for _, row in sample.iterrows():
        should_flag = False
        for col in ['PM2.5', 'SO2', 'NOx']:
            val = row[col]
            if pd.notna(val):
                limit = limit_lookup.get((col, row['Source_Type']))
                if limit and val > limit:
                    should_flag = True
        expected = 'EXCEEDANCE' if should_flag else 'OK'
        if row['Exceedance_Flag'] != expected:
            mismatches += 1
    
    check('R19', 'Exceedance flags are correct (spot-check)',
          mismatches == 0,
          f'Checked {len(sample)} rows, mismatches: {mismatches}')

✅ R19      | Exceedance_Flag column exists — Values: {'EXCEEDANCE': 16983, 'OK': 11817}
✅ R19      | Exceedance flags are correct (spot-check) — Checked 100 rows, mismatches: 0


In [497]:
# ═══ R20: Audit hash column exists ═══
has_hash = 'Audit_Hash' in df.columns
check('R20', 'Audit_Hash column exists',
      has_hash,
      f'Sample: {df["Audit_Hash"].iloc[0][:16]}...' if has_hash else 'MISSING')

if has_hash:
    # verify hash length (SHA-256 = 64 hex chars)
    hash_lengths = df['Audit_Hash'].str.len().unique()
    check('R20', 'All hashes are 64 characters (SHA-256)',
          len(hash_lengths) == 1 and hash_lengths[0] == 64,
          f'Hash lengths: {hash_lengths}')

✅ R20      | Audit_Hash column exists — Sample: d71814caf8869de7...
✅ R20      | All hashes are 64 characters (SHA-256) — Hash lengths: [64]


---
## Audit Log Validation

In [498]:
# ═══ Audit log completeness ═══
check('AUDIT', 'Audit log is non-empty',
      len(audit) > 0,
      f'{len(audit)} entries')

# check that all expected rules appear in the log
logged_rules = set(audit['Rule'].unique())
expected_rules = {'R1', 'R2', 'R3', 'R4', 'R5', 'R7', 'R8', 'R9',
                  'R10', 'R11', 'R12', 'R14', 'R17', 'TS_FORMAT'}
missing_rules = expected_rules - logged_rules

check('AUDIT', 'All cleaning rules appear in audit log',
      len(missing_rules) == 0,
      f'Missing: {missing_rules}' if missing_rules else f'Rules logged: {sorted(logged_rules)}')

# check audit log has required columns
audit_cols = {'Record_ID', 'Column', 'Old', 'New', 'Rule'}
# audit_cols = {'Record_ID', 'Column', 'Old', 'New', 'Rule'}
audit_cols_present = audit_cols.issubset(set(audit.columns))
check('AUDIT', 'Audit log has all required columns',
      audit_cols_present,
      f'Columns: {list(audit.columns)}')

# breakdown by rule
print('\nAudit log breakdown by rule:')
print(audit['Rule'].value_counts().to_string())

✅ AUDIT    | Audit log is non-empty — 124306 entries
✅ AUDIT    | All cleaning rules appear in audit log — Rules logged: ['R1', 'R10', 'R11', 'R12', 'R14', 'R17', 'R2', 'R3', 'R4', 'R4b', 'R5', 'R7', 'R8', 'R9', 'TS_FORMAT']
✅ AUDIT    | Audit log has all required columns — Columns: ['Record_ID', 'Column', 'Old', 'New', 'Rule']

Audit log breakdown by rule:
Rule
R7           73900
R14          10656
R8            9300
R3            7742
R12           4114
R5            4061
R11           4061
R2            2623
R9            2415
TS_FORMAT     1874
R17           1292
R1             797
R10            796
R4             514
R4b            161


---
## Data Quality Summary

In [499]:
# ═══ Overall statistics ═══
print('\n' + '=' * 60)
print('  DATA QUALITY SUMMARY')
print('=' * 60)
print(f'  Total rows:         {len(df):,}')
print(f'  Total columns:      {len(df.columns)}')
print(f'  Audit log entries:  {len(audit):,}')
print(f'  Date range:         {df["TS"].min()} to {df["TS"].max()}')
print(f'  Unit values:        {df["Unit"].unique()}')
print(f'  Status values:      {sorted(df["Status"].unique())}')
print('\nPollutant ranges (after cleaning):')
for col in ['PM2.5', 'SO2', 'NOx']:
    vals = df[col].dropna()
    print(f'  {col:6s}: min={vals.min():.1f}, max={vals.max():.1f}, mean={vals.mean():.1f}, nulls={df[col].isna().sum()}')
print('=' * 60)


  DATA QUALITY SUMMARY
  Total rows:         28,800
  Total columns:      14
  Audit log entries:  124,306
  Date range:         2025-01-01 00:00:00 to 2025-01-30 23:45:00
  Unit values:        ['ug/m3']
  Status values:      ['FAULT', 'GAP_FILLED', 'MAINT', 'OK', 'UNKNOWN']

Pollutant ranges (after cleaning):
  PM2.5 : min=0.3, max=2499.9, mean=141.8, nulls=4333
  SO2   : min=0.8, max=2499.9, mean=146.0, nulls=4362
  NOx   : min=0.8, max=2550.8, mean=142.7, nulls=4365


---
## Final Scorecard

In [500]:
# ═══ Final Results ═══
results_df = pd.DataFrame(results)
passed = (results_df['Status'] == 'PASS').sum()
failed = (results_df['Status'] == 'FAIL').sum()
total = len(results_df)

print('\n' + '=' * 60)
print('  VALIDATION SCORECARD')
print('=' * 60)
print(f'  Total checks:  {total}')
print(f'  \u2705 Passed:     {passed}')
print(f'  \u274c Failed:     {failed}')
print(f'  Score:         {passed}/{total} ({passed/total*100:.0f}%)')
print('=' * 60)

if failed > 0:
    print('\n\u274c FAILED CHECKS:')
    for _, row in results_df[results_df['Status'] == 'FAIL'].iterrows():
        print(f'  {row["Rule"]:8s} | {row["Check"]} — {row["Detail"]}')
else:
    print('\n\u2705 ALL CHECKS PASSED — Data cleaning validated successfully!')

# save results
results_df.to_csv(f'{DATA_DIR}/validation_results.csv', index=False)
print(f'\nResults saved to: {DATA_DIR}/validation_results.csv')


  VALIDATION SCORECARD
  Total checks:  44
  ✅ Passed:     44
  ❌ Failed:     0
  Score:         44/44 (100%)

✅ ALL CHECKS PASSED — Data cleaning validated successfully!

Results saved to: ../cleaned/full/validation_results.csv
